# 03 - Data Structures and Algorithms (Java)

This notebook builds the same fault-stack / action-queue / CAN-ID-lookup example from `concept.md`, in Java, then runs the same Big-O demo comparing linear search against a hashmap lookup. Read `concept.md` first if you haven't.

## Imports

In [1]:
import java.util.ArrayDeque;
import java.util.Deque;
import java.util.HashMap;
import java.util.List;
import java.util.ArrayList;
import java.util.Map;

## Stack: Recent Controller Faults

Java's `Deque` interface (via the `ArrayDeque` implementation) covers both stacks and queues. Java doesn't have a separate dedicated `Stack` type worth using today (the old `java.util.Stack` class exists but is legacy and slower). `push()`/`pop()` on a `Deque` operate on the front, giving Last In, First Out.

In [2]:
Deque<String> faultStack = new ArrayDeque<>();

faultStack.push("CAN timeout: device 12");
faultStack.push("Brownout detected");
faultStack.push("CAN timeout: device 7");

System.out.println("Most recent fault first:");
while (!faultStack.isEmpty()) {
    System.out.println(" - " + faultStack.pop());
}

Most recent fault first:


 - CAN timeout: device 7


 - Brownout detected


 - CAN timeout: device 12


Same result as the Python notebook: the last fault pushed ("CAN timeout: device 7") is the first one popped.

## Queue: Autonomous Action Sequence

The exact same `ArrayDeque` object can be used as a queue instead of a stack, just by calling different methods: `offer()` to enqueue at the back, `poll()` to dequeue from the front.

In [3]:
Deque<String> actionQueue = new ArrayDeque<>();
actionQueue.offer("drive forward");
actionQueue.offer("intake");
actionQueue.offer("shoot");
actionQueue.offer("drive back");

System.out.println("Executing in queued order:");
while (!actionQueue.isEmpty()) {
    System.out.println(" - " + actionQueue.poll());
}

Executing in queued order:


 - drive forward


 - intake


 - shoot


 - drive back


Same underlying class (`ArrayDeque`), completely different behavior, purely because of which methods we called. This is a good reminder that "stack" and "queue" are really about *which operations you use*, not some fundamentally different piece of hardware.

## Hashmap: CAN ID to Device Name

Java's `HashMap` is the direct equivalent of Python's `dict`: key-value pairs, average O(1) lookup by key.

In [4]:
Map<Integer, String> canIdToName = new HashMap<>();
canIdToName.put(1, "Front Left Drive");
canIdToName.put(2, "Front Right Drive");
canIdToName.put(3, "Back Left Drive");
canIdToName.put(4, "Back Right Drive");
canIdToName.put(12, "Intake Roller");

System.out.println(canIdToName.get(12));
System.out.println(canIdToName.getOrDefault(99, "<unknown device>"));

Intake Roller


<unknown device>


`getOrDefault(key, default)` is Java's equivalent of Python's `.get(key, default)`. It is a safe lookup that doesn't throw or return `null` for a missing key.

## Tree: The Same CAN ID Table, Sorted

Java's `HashMap` doesn't keep entries in any particular order; iterating over `canIdToName` gives them back in whatever order the hash table happens to store them, not sorted by key. A **binary search tree (BST)** keeps entries sorted by key while still supporting fast lookup: every node holds a key/value pair plus a link to a left child (smaller keys) and a right child (larger keys). Java has no BST in the standard collections the way it has `HashMap`, so we build one by hand, same spirit as `FaultStack` in `03`'s C++ files.

In [5]:
class DeviceTree {
    private static class Node {
        int canId;
        String name;
        Node left;
        Node right;

        Node(int canId, String name) {
            this.canId = canId;
            this.name = name;
        }
    }

    private Node root;

    void insert(int canId, String name) {
        root = insertNode(root, canId, name);
    }

    private Node insertNode(Node node, int canId, String name) {
        if (node == null) {
            return new Node(canId, name);
        }
        if (canId < node.canId) {
            node.left = insertNode(node.left, canId, name);
        } else if (canId > node.canId) {
            node.right = insertNode(node.right, canId, name);
        } else {
            node.name = name; // Same ID seen again -- update rather than duplicate.
        }
        return node;
    }

    String find(int canId) {
        Node current = root;
        while (current != null) {
            if (canId == current.canId) {
                return current.name;
            }
            current = canId < current.canId ? current.left : current.right;
        }
        return null;
    }

    void forEachInOrder(java.util.function.BiConsumer<Integer, String> visit) {
        forEachNode(root, visit);
    }

    private void forEachNode(Node node, java.util.function.BiConsumer<Integer, String> visit) {
        if (node == null) {
            return;
        }
        forEachNode(node.left, visit);
        visit.accept(node.canId, node.name);
        forEachNode(node.right, visit);
    }
}

Only these five devices go into the tree, inserted in arbitrary (not sorted) order.This is deliberately **not** the 10,000-entry table used in the Big-O demo below. Inserting keys that are already sorted into a plain BST like this one builds a completely lopsided tree (every node with only a right child), which defeats O(log n) entirely; a real tree implementation would rebalance itself instead, which is out of scope here.

In [6]:
DeviceTree deviceTree = new DeviceTree();
deviceTree.insert(4, "Back Right Drive");
deviceTree.insert(1, "Front Left Drive");
deviceTree.insert(12, "Intake Roller");
deviceTree.insert(2, "Front Right Drive");
deviceTree.insert(3, "Back Left Drive");

System.out.println(deviceTree.find(12));
String maybeName = deviceTree.find(99);
System.out.println(maybeName != null ? maybeName : "<unknown device>");

System.out.println("\nAll devices, sorted by CAN ID (in-order tree walk):");
deviceTree.forEachInOrder((id, name) -> System.out.println(" - " + id + ": " + name));

Intake Roller


<unknown device>



All devices, sorted by CAN ID (in-order tree walk):


 - 1: Front Left Drive


 - 2: Front Right Drive


 - 3: Back Left Drive


 - 4: Back Right Drive


 - 12: Intake Roller


Notice the in-order walk prints every device sorted by CAN ID (1, 2, 3, 4, 12) even though we inserted them out of order (4, 1, 12, 2, 3).

## Big-O in Practice: Linear Search vs. Binary Search vs. Hashmap Lookup

Same experiment as the Python notebook: a big table of (id, name) pairs, and a count of how many comparisons a linear scan, a binary search, and a hashmap lookup each need to find an ID at different positions.

In [ ]:
record TableEntry(int id, String name) {}

int linearSearchComparisons(List<TableEntry> entries, int targetId) {
    int comparisons = 0;
    for (TableEntry entry : entries) {
        comparisons++;
        if (entry.id() == targetId) {
            return comparisons;
        }
    }
    return comparisons;
}

int binarySearchComparisons(List<TableEntry> sortedEntries, int targetId) {
    int comparisons = 0;
    int low = 0;
    int high = sortedEntries.size() - 1;
    while (low <= high) {
        comparisons++;
        int mid = low + (high - low) / 2;
        int midId = sortedEntries.get(mid).id();
        if (midId == targetId) {
            return comparisons;
        } else if (midId < targetId) {
            low = mid + 1;
        } else {
            high = mid - 1;
        }
    }
    return comparisons;
}

List<TableEntry> bigTable = new ArrayList<>();
Map<Integer, String> bigMap = new HashMap<>();
for (int i = 0; i < 10_000; i++) {
    bigTable.add(new TableEntry(i, "device-" + i));
    bigMap.put(i, "device-" + i);
}
// bigTable is already sorted by id (built in order 0..9999), which is
// exactly what binarySearchComparisons requires.

for (int target : new int[] {5, 5_000, 9_999}) {
    int linearComparisons = linearSearchComparisons(bigTable, target);
    int binaryComparisons = binarySearchComparisons(bigTable, target);
    System.out.println("id=" + target + " -> linear: " + linearComparisons
        + " comparisons, binary search: " + binaryComparisons + " comparisons");
}

System.out.println();
System.out.println("hashmap lookup for id=5:    " + bigMap.get(5));
System.out.println("hashmap lookup for id=9999: " + bigMap.get(9999));

Same conclusion as the Python notebook. The comparison count for linear search grows directly with the target's position, binary search barely grows at all (roughly log₂(n): a dozen or so comparisons whether the target is at position 5,000 or 9,999), and the hashmap lookups do essentially the same amount of work either way.

One thing you may notice is that for `id=5`, the linear scan can actually take *fewer* comparisons than binary search on that one specific call. That's not a contradiction, because a linear scan's best case (an early match) is cheap no matter how big the data gets. Remember, Big-O describes the worst case *as data keeps growing*, not a promise that the asymptotically-better algorithm wins every single call. Binary search's guarantee is that it never gets much worse than ~log₂(n) regardless of where the target sits; linear search's worst case keeps getting worse as the table grows.

## Try It Yourself

No solutions are provided. These are meant to be worked through on your own or with a mentor or another student.

1. Add a `peek()` call to the fault stack example that looks at the most recent fault *without* removing it.
2. Modify `linearSearchComparisons` to also work correctly for a target that isn't in `bigTable` at all, and confirm it returns `bigTable.size()`.
3. Time (with `System.nanoTime()`) an actual `bigMap.get(9999)` versus an actual `linearSearchComparisons(bigTable, 9999)` call, over many repetitions, and see if the wall-clock gap matches what the comparison counts predicted.
4. Add a `remove(canId)` method to `DeviceTree`. (Hint: this is the hardest of the four — removing a node with two children means you need to pick a replacement, usually the smallest node in its right subtree, without breaking the BST ordering.)

In [8]:
// Your code here
